# HVAC v19s retrain — direct upload + Drive backup (bulletproof)

Uses the **940 MB** bundle `hvac_v19s_tiled_colab.zip` (10,000 tiles, ALL air-device tiles kept — the one meant to BEAT v10).

### Order of operations
1. `Runtime -> Change runtime type -> GPU`, run cells 1-2.
2. Click the **folder icon** (left sidebar) -> **drag `hvac_v19s_tiled_colab.zip` in** (940 MB, ~5-20 min upload). Wait for the spinner to finish.
3. Run cell 3 (unzip), cell 4 (train ~60-90 min).
4. Run **cell 5 (Drive backup)** FIRST — this is the reliable copy. Then cell 6 (browser download) as a convenience.

### Why two save paths
Last run the browser download silently failed (`.pt` is blocked until you click **Keep**, and the file never hit disk). Cell 5 writes the weight straight to **your Google Drive** so it can't be lost. Cell 6 also tries the direct download. Use whichever lands.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE -> Runtime > Change runtime type > GPU, then Run all again')

In [ ]:
!pip -q install ultralytics==8.4.51

In [ ]:
# Drag the zip into the left file panel first (recommended), then run this.
from google.colab import files
import zipfile, os, glob
found = glob.glob('/content/*.zip')
if not found:
    up = files.upload()              # picker fallback if you didn't drag it in
    found = ['/content/' + k for k in up]
zname = found[0]
print('using:', zname, f'({os.path.getsize(zname)/1e6:.0f} MB)')
os.makedirs('/content/hvac', exist_ok=True)
with zipfile.ZipFile(zname) as z:
    z.extractall('/content/hvac')
os.chdir('/content/hvac')
print('ready:', os.getcwd(), sorted(os.listdir('.'))[:8])

In [ ]:
# Train. ~60-90 min on a T4 for 10k tiles. (Drop --epochs to 40 for a faster first pass.)
# run-name v19s so the promoted weight is models/hvac_yolov8s_v19s.pt.
!python train_v11.py --data-yaml yolo_dataset_v19s_tiled/data.yaml \
    --epochs 60 --device 0 --batch 16 --imgsz 640 --run-name v19s

In [ ]:
# === CELL 5: DRIVE BACKUP (the reliable save — run this FIRST) ===
# Mounts your Google Drive and copies the trained weight to
# MyDrive/hvac_models/. A click-through auth pops up the first time — sign in
# as the SAME Google account you use for Drive, click Allow. No phone OTP.
import glob, os, shutil
from google.colab import drive
drive.mount('/content/drive')

cands = glob.glob('models/hvac_yolov8s_v19s.pt') + glob.glob('runs/**/weights/best.pt', recursive=True)
assert cands, 'No weights found - check the training cell output above.'
src = cands[0]
dest_dir = '/content/drive/MyDrive/hvac_models'
os.makedirs(dest_dir, exist_ok=True)
dest = os.path.join(dest_dir, 'hvac_yolov8s_v19s.pt')
shutil.copy2(src, dest)
print(f'BACKED UP to Drive: {dest}  ({os.path.getsize(dest)/1e6:.1f} MB)')
print('--> Find it in Google Drive at: MyDrive/hvac_models/hvac_yolov8s_v19s.pt')
print('--> Download it from Drive to your PC Downloads, then tell Claude.')

In [ ]:
# === CELL 6: DIRECT BROWSER DOWNLOAD (convenience fallback) ===
# If a '...can harm your computer' bar appears, click KEEP or the file never
# saves. Then check it landed in your PC's Downloads folder.
import glob
from google.colab import files
cands = glob.glob('models/hvac_yolov8s_v19s.pt') + glob.glob('runs/**/weights/best.pt', recursive=True)
assert cands, 'No weights found.'
print('downloading', cands[0], '-- click KEEP if Chrome warns about the .pt file')
files.download(cands[0])